# Optimización por Descenso del Gradiente

## Librerías y Tipado

In [ ]:
# Módulos
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation  # Animaciones auxiliares

# Tipos
from numpy.typing import NDArray
from typing import Callable, TypedDict
type Array = NDArray[np.float64]
class Configuration(TypedDict):
    rate: float
    max_iterations: int
    initial_position: Array
    function: Callable[[Array], float]
    gradient_function: Callable[[Array], Array]

## Algoritmo

In [ ]:
LIMITS = (-5.0, 5.0)
"""
Los límites `a` y `b` para la región [a, b]² donde se genera la posición
inicial.
"""

def get_initial_position(size: int):
    """Genera un arreglo de números aleatorios.
    
    Los valores del arreglo tienen distribución uniforme con limites dados por
    `LIMITS`.
    """
    return np.random.uniform(*LIMITS, size)

def descend(
        position: Array,
        gradient_function: Callable[[Array], Array],
        rate: float
):
    """Genera una mejor posición descendiendo desde la posición actual.
    
    Para determinar la nueva posición, se evalúa `gradient_function` en
    `position`, y se toma un paso en la dirección opuesta del gradiente
    obtenido, con su magnitud escalada por `rate`.

    Se retorna `(p_n, g, c)`, donde `p_n` es la nueva posición, `g` es el
    gradiente calculado y `c` es el paso tomado.
    """
    gradient = gradient_function(position)
    change = -rate * gradient
    return position + change, gradient, change

def run(
        initial_position: Array,
        function: Callable[[Array], float],
        gradient_function: Callable[[Array], Array],
        rate: float,
        max_iterations: int
):
    """Implementación del algoritmo del descenso del gradiente.
    
    De manera repetida, evalúa el gradiente en un punto y desciende haciendo
    uso de la función `descend` hasta alcanzar el máximo de iteraciones."""
    position = initial_position
    print('Posición Inicial:', position)
    print('Valor Función Objetivo:', function(position), '\n')

    for i in range(max_iterations):
        position, gradient, change = descend(
            position, gradient_function, rate
        )
        print('Iteración', i + 1)
        print(f'\t{gradient=}')
        print(f'\t{change=}')
        print(f'\t{position=}')

    print('\nPosición Final:', position)
    print('Valor Función Objetivo:', function(position))

# Animación auxiliar para casos 2D
def plot_animation(
        samples: int,
        function: Callable[[Array, Array], Array],
        curves: int,
        configuration: Configuration,
        output_name: str,
        duration: float = 5,
):
    """Genera una animación del cambio de la solución sobre las curvas de nivel.
    
    Se genera una cuadrícula de puntos con `samples` puntos de muestra sobre
    ambos ejes coordenados, a partir de la cual se grafican las curvas de nivel
    de la función especificada (usando `num_curves` curvas), y se anima el
    cambio del punto solución a lo largo del tiempo con los parámetros dados en
    `configuration`. La animación
    resultante se guarda en el directorio actual como un video mp4 de `duration`
    segundos con el nombre `output_name`."""
    X, Y = np.meshgrid(*([np.linspace(*LIMITS, samples)]*2))
    fig, ax = plt.subplots()
    ax.contour(X, Y, function(X, Y), curves)
    point = ax.plot(*configuration['initial_position'], 'ro')[0]

    def update(_):
        """Actualiza un punto graficado usando el gradiente descendiente.
        
        Obtiene el punto actual en el gráfico creado y aplica una iteración
        del algoritmo para retornar el punto del siguiente frame.
        """
        point.set_data(*descend(
            np.array(point.get_data()),
            configuration['gradient_function'],
            configuration['rate']
        )[0])
        return [point]

    anim = animation.FuncAnimation(
        fig,
        func=update,
        frames=configuration['max_iterations'],
        interval=(
            1000*duration / configuration['max_iterations']
        )
    )

    anim.save(output_name)

## Función de Rosenbrock

La [función generalizada de Rosenbrock](https://docs.scipy.org/doc/scipy-0.14.0/reference/tutorial/optimize.html#unconstrained-minimization-of-multivariate-scalar-functions-minimize) se expresa como $
f(\mathbf{x}) = \sum _{i=1}^{N-1}\left( 100\left( x_{i} -x_{i-1}^{2}\right)^{2} +( 1-x_{i-1})^{2}\right)
$.

Obtenemos una expresión para el i-ésimo componente del gradiente de Rosenbrock $\nabla f(\mathbf{x})$ por casos:

$$\begin{aligned}
\frac{\partial f(\mathbf{x})}{\partial x_{0}} & =\frac{\partial }{\partial x_{0}}\left( 100\left( x_{1} -x_{0}^{2}\right)^{2} +( 1-x_{0})^{2} +\cdots \right)\\
 & =-400x_{0}\left( x_{1} -x_{0}^{2}\right) -2( 1-x_{0})
\end{aligned} \tag{1}$$

$$\begin{aligned}
\frac{\partial f(\mathbf{x})}{\partial x_{N-1}} & =\frac{\partial }{\partial x_{N-1}}\left( \cdots +100\left( x_{N-1} -x_{N-2}^{2}\right)^{2} +( 1-x_{N-2})^{2}\right)\\
 & =200\left( x_{N-1} -x_{N-2}^{2}\right)
\end{aligned} \tag{2}$$

$$\begin{aligned}
\forall k\in \{1,2,\dotsc ,N-2\} ,\frac{\partial f(\mathbf{x})}{\partial x_{k}} & =\frac{\partial }{\partial x_{k}}\sum _{i=1}^{N-1}\left( 100\left( x_{i} -x_{i-1}^{2}\right)^{2} +( 1-x_{i-1})^{2}\right)\\
 & \begin{align*}
=\frac{\partial }{\partial x_{k}}\Bigl[ \cdots + & \left( 100\left( x_{k} -x_{k-1}^{2}\right)^{2} +( 1-x_{k-1})^{2}\right) +\\
 & \left( 100\left( x_{k+1} -x_{k}^{2}\right)^{2} +( 1-x_{k})^{2}\right) +\cdots \Bigr]
\end{align*}\\
 & =200\left( x_{k} -x_{k-1}^{2}\right) -400x_{k}\left( x_{k+1} -x_{k}^{2}\right) -2( 1-x_{k})\\
 & =400x_{k}^{3} +( 202-400x_{k+1}) x_{k} -200x_{k-1}^{2} -2
\end{aligned} \tag{3}$$

Así, para el caso de 2 variables, utilizamos $\begin{cases}
N = 2, \\
x_0 = x, \\
x_1 = y,
\end{cases}$ donde no aplicaría $(3)$, y para 3 variables, $\begin{cases}
N = 3, \\
x_0 = x, \\
x_1 = y, \\
x_2 = z.
\end{cases}$

In [ ]:
def rosenbrock(x: Array):
    """Calcula la función de rosenbrock en un punto x con `N` dimensiones."""
    N = len(x)
    return sum(
        100 * (x[i] - x[i-1]**2)**2 + (1 - x[i-1])**2
        for i in range(1, N)
    )

def rosenbrock_gradient(x: Array):
    """Calcula el gradiente de rosenbrock en un punto x con `N` dimensiones."""
    N = len(x)
    return np.array([
        -400*x[0]*(x[1] - x[0]**2) - 2*(1 - x[0]) if k == 0 else
        200 * (x[N-1] - x[N-2]**2) if k == N-1 else
        400*x[k] + (202 - 400*x[k+1])*x[k] - 200*x[k-1]**2 - 2
        for k in range(N)
    ])

### 2D

#### Optimización

In [ ]:
rosenbrock_2d_configuration: Configuration = {
    'rate': 0.0001,
    'max_iterations': 1000,
    'initial_position': get_initial_position(2),
    'function': rosenbrock,
    'gradient_function': rosenbrock_gradient,
}

run(**rosenbrock_2d_configuration)

#### Animación Auxiliar

In [ ]:
plot_animation(
    samples=50,
    function=lambda X, Y: 100*(Y - X**2)**2 + (1 - X)**2,
    curves=100,
    configuration=rosenbrock_2d_configuration,
    output_name='rosen_2d_grad.mp4'
)

### 3D

In [ ]:
rosenbrock_3d_configuration = {
    'initial_position': get_initial_position(3),
    'rate': 0.000_1,
    'max_iterations': 10_000,
    'function': rosenbrock,
    'gradient_function': rosenbrock_gradient
}

run(**rosenbrock_3d_configuration)

## [Función de Schwefel](https://www.sfu.ca/~ssurjano/schwef.html)

$$f(\mathbf{x}) =418.9829d-\sum _{i=1}^{d} x_{i}\sin{\sqrt{|x_{i}|}}$$

$$\begin{aligned}
    \frac{\partial f}{\partial x_{k}} & = \frac{\partial }{\partial x_{k}}
        \left( -\sum _{i=1}^{d} x_{i}\sin{\sqrt{|x_{i}|}}\right) \\
    & = -\frac{\partial}{\partial x_{k}} x_{k} \sin{\sqrt{|x_{k}|}} \\
    & = -x_{k}\frac{\partial }{\partial x_{k}}
        \left(\sin{\sqrt{|x_{k}|}}\right) -\sin{\sqrt{|x_{k}|}} \\
    & \begin{array}{l}
        =-\frac{\cos{\sqrt{|x_{k}|}}}{2 \sqrt{|x_{k}|}} s_{k} x_{k}
            - \sin{\sqrt{|x_{k}|}} \\
    \end{array}
\end{aligned}$$

donde $s_{k} = \frac{\partial}{\partial x_{k}} |x_{k}| = \begin{cases}
    1, & x > 0,\\
    -1, & x < 0.
\end{cases}$

Algo a resaltar (y el divisor $2 \sqrt{|x_k|}$ lo hace evidente) es que esta
derivación no es válida para $x_k = 0$. Si bien esto no nos preocupa para la
implementación del algoritmo (pues es altamente improbable obtener una
coordenada exactamente igual a 0), sí indica que el proceso de optimización
puede ser problemático si en algún momento nos encontramos cerca de las rectas
$x = 0$ y $y = 0$.

In [ ]:
def schwefel(x: Array):
    d = len(x)
    return 418.9829*d - np.sum(x * np.sin(np.sqrt(np.abs(x))))

def schwefel_gradient(x: Array):
    sq_abs_x = np.sqrt(np.abs(x))
    return -np.cos(sq_abs_x)/(2 * sq_abs_x) - np.sin(sq_abs_x)

### 2D

#### Optimización

In [ ]:
schwefel_2d_configuration: Configuration = {
    'initial_position': get_initial_position(2),
    'rate': 0.001,
    'max_iterations': 10_000,
    'function': schwefel,
    'gradient_function': schwefel_gradient
}

run(**schwefel_2d_configuration)

#### Animación Auxiliar

In [ ]:
plot_animation(
    samples=10,
    function=lambda X, Y: 418.9829*2 - X*np.sin(np.sqrt(np.abs(X))) - Y*np.sin(np.sqrt(np.abs(Y))),
    configuration=schwefel_2d_configuration,
    curves=10,
    output_name='schwefel_2d_grad.mp4'
)

### 3D

In [ ]:
schwefel_2d_configuration = {
    'initial_position': get_initial_position(3),
    'rate': 0.001,
    'max_iterations': 10_000,
    'function': schwefel,
    'gradient_function': schwefel_gradient
}

run(**schwefel_2d_configuration)